In [ ]:
import os
DATA_ROOT = os.environ.get('WRH_DATA_ROOT', 'G:/WRH_data')  # local data folder; see README
# ==================== FIGURE A · 4 standalone maps ====================
import os, numpy as np, pandas as pd, matplotlib.pyplot as plt
from matplotlib.colors import BoundaryNorm, ListedColormap
import cartopy.crs as ccrs, cartopy.feature as cfeature

MD = pd.read_csv(rf'{DATA_ROOT}/new_climate/_proj_global/tables/map_days.csv')
NLAT, NLON = 90, 180
LAT_E, LON_E = np.arange(-90, 91, 2.0), np.arange(-180, 181, 2.0)
_BAD = (~MD['good'].values.astype(bool)) | (~MD['in_domain'].values.astype(bool)) \
       | MD['masked'].values.astype(bool)

BINS = [-40, -20, -10, -5, -1, 1, 5, 10, 20, 40]
CMAP = ListedColormap(np.vstack([plt.cm.RdBu_r(np.linspace(.03, .38, 4)),
                                 [[1, 1, 1, 1]],
                                 plt.cm.RdBu_r(np.linspace(.62, .97, 4))]))
CMAP.set_bad('#c8ccce')
NORM  = BoundaryNorm(BINS, CMAP.N)
CBLAB = 'Change in exceedance days per year (2081\u20132100 minus 2021\u20132040)'
TTL   = {('wave','ssp126'): 'Change under SSP1-2.6 ($H_S$)',
         ('wave','ssp585'): 'Change under SSP5-8.5 ($H_S$)',
         ('wind','ssp126'): 'Change under SSP1-2.6 ($U_{10}$)',
         ('wind','ssp585'): 'Change under SSP5-8.5 ($U_{10}$)'}

def field(kind, ssp, sl):
    v = (MD[f'{kind}_ACCESS_{ssp}_{sl}'].values + MD[f'{kind}_ECEARTH_{ssp}_{sl}'].values) / 2
    g = np.full(NLAT * NLON, np.nan); g[MD['flat_idx'].values] = np.where(_BAD, np.nan, v)
    return g.reshape(NLAT, NLON)

def map_panel(kind, ssp, cbar=False, figsize=(5.8, 3.4)):
    fig = plt.figure(figsize=figsize, dpi=180)
    ax  = fig.add_subplot(111, projection=ccrs.Robinson(central_longitude=0))
    Z   = field(kind, ssp, 'end') - field(kind, ssp, 'near')
    im  = ax.pcolormesh(LON_E, LAT_E, np.ma.masked_invalid(Z), cmap=CMAP, norm=NORM,
                        transform=ccrs.PlateCarree(), shading='flat', zorder=2)
    ax.add_feature(cfeature.LAND, facecolor='#d9d9d9', edgecolor='none', zorder=3)
    ax.coastlines(linewidth=.2, color='#a9a9a9', zorder=4)
    ax.set_extent([-180, 180, -60, 60], crs=ccrs.PlateCarree())
    ax.spines['geo'].set_linewidth(.5); ax.spines['geo'].set_edgecolor('#7a7a7a')
    gl = ax.gridlines(draw_labels=True, linewidth=0, zorder=1,
                      xlocs=np.arange(-180, 181, 60), ylocs=[-30, 0, 30])
    gl.top_labels = gl.right_labels = False
    gl.xlabel_style = gl.ylabel_style = {'size': 7.5, 'color': '#333333'}
    ax.set_title(TTL[(kind, ssp)], fontsize=11, pad=7)
    if cbar:
        cb = fig.colorbar(im, ax=ax, orientation='horizontal', fraction=.06,
                          pad=.09, aspect=34, ticks=BINS, extend='both')
        cb.set_label(CBLAB, fontsize=8.5); cb.ax.tick_params(labelsize=7.5)
        cb.outline.set_linewidth(.5)
    plt.show()

def map_colorbar_only(figsize=(9.2, .85)):
    """Standalone colour bar, so a single shared bar can be placed when assembling panels"""
    fig, ax = plt.subplots(figsize=figsize, dpi=180)
    cb = fig.colorbar(plt.cm.ScalarMappable(norm=NORM, cmap=CMAP), cax=ax,
                      orientation='horizontal', ticks=BINS, extend='both')
    cb.set_label(CBLAB, fontsize=10); cb.ax.tick_params(labelsize=9)
    cb.outline.set_linewidth(.5)
    plt.show()

for _k, _s in [('wave','ssp126'), ('wave','ssp585'), ('wind','ssp126'), ('wind','ssp585')]:
    map_panel(_k, _s)
map_colorbar_only()


In [ ]:
# ==================== FIGURE A · 4 maps × with/without labels ====================
import os, numpy as np, pandas as pd, matplotlib.pyplot as plt
from matplotlib.colors import BoundaryNorm, ListedColormap
import cartopy.crs as ccrs, cartopy.feature as cfeature

DPI = 500
MD  = pd.read_csv(rf'{DATA_ROOT}/new_climate/_proj_global/tables/map_days.csv')
NLAT, NLON = 90, 180
LAT_E, LON_E = np.arange(-90, 91, 2.0), np.arange(-180, 181, 2.0)
_BAD = (~MD['good'].values.astype(bool)) | (~MD['in_domain'].values.astype(bool)) \
       | MD['masked'].values.astype(bool)

BINS = [-40, -20, -10, -5, -1, 1, 5, 10, 20, 40]
CMAP = ListedColormap(np.vstack([plt.cm.RdBu_r(np.linspace(.03, .38, 4)),
                                 [[1, 1, 1, 1]],
                                 plt.cm.RdBu_r(np.linspace(.62, .97, 4))]))
CMAP.set_bad('#c8ccce')
NORM  = BoundaryNorm(BINS, CMAP.N)
CBLAB = 'Change in exceedance days per year (2081\u20132100 minus 2021\u20132040)'
TTL   = {('wave','ssp126'): 'Change under SSP1-2.6 ($H_S$)',
         ('wave','ssp585'): 'Change under SSP5-8.5 ($H_S$)',
         ('wind','ssp126'): 'Change under SSP1-2.6 ($U_{10}$)',
         ('wind','ssp585'): 'Change under SSP5-8.5 ($U_{10}$)'}

def field(kind, ssp, sl):
    v = (MD[f'{kind}_ACCESS_{ssp}_{sl}'].values + MD[f'{kind}_ECEARTH_{ssp}_{sl}'].values) / 2
    g = np.full(NLAT * NLON, np.nan); g[MD['flat_idx'].values] = np.where(_BAD, np.nan, v)
    return g.reshape(NLAT, NLON)

def map_panel(kind, ssp, annot=True, cbar=False, figsize=(5.8, 3.4)):
    fig = plt.figure(figsize=figsize, dpi=DPI)
    ax  = fig.add_subplot(111, projection=ccrs.Robinson(central_longitude=0))
    Z   = field(kind, ssp, 'end') - field(kind, ssp, 'near')
    im  = ax.pcolormesh(LON_E, LAT_E, np.ma.masked_invalid(Z), cmap=CMAP, norm=NORM,
                        transform=ccrs.PlateCarree(), shading='flat', zorder=2)
    ax.add_feature(cfeature.LAND, facecolor='#d9d9d9', edgecolor='none', zorder=3)
    ax.coastlines(linewidth=.2, color='#a9a9a9', zorder=4)
    ax.set_extent([-180, 180, -60, 60], crs=ccrs.PlateCarree())
    ax.spines['geo'].set_linewidth(.5); ax.spines['geo'].set_edgecolor('#7a7a7a')
    gl = ax.gridlines(draw_labels=annot, linewidth=0, zorder=1,
                      xlocs=np.arange(-180, 181, 60), ylocs=[-30, 0, 30])
    if annot:
        gl.top_labels = gl.right_labels = False
        gl.xlabel_style = gl.ylabel_style = {'size': 7.5, 'color': '#333333'}
        ax.set_title(TTL[(kind, ssp)], fontsize=11, pad=7)
    if cbar:
        cb = fig.colorbar(im, ax=ax, orientation='horizontal', fraction=.06,
                          pad=.09, aspect=34, ticks=BINS, extend='both')
        if annot: cb.set_label(CBLAB, fontsize=8.5)
        cb.ax.tick_params(labelsize=7.5); cb.outline.set_linewidth(.5)
    plt.show()

def map_colorbar_only(figsize=(9.2, .85)):
    fig, ax = plt.subplots(figsize=figsize, dpi=DPI)
    cb = fig.colorbar(plt.cm.ScalarMappable(norm=NORM, cmap=CMAP), cax=ax,
                      orientation='horizontal', ticks=BINS, extend='both')
    cb.set_label(CBLAB, fontsize=10); cb.ax.tick_params(labelsize=9)
    cb.outline.set_linewidth(.5)
    plt.show()

for _k, _s in [('wave','ssp126'), ('wave','ssp585'), ('wind','ssp126'), ('wind','ssp585')]:
    map_panel(_k, _s, annot=True)
    map_panel(_k, _s, annot=False)
map_colorbar_only()


In [ ]:
# ==================== FIGURE B · 4 line plots × with/without labels (no thin background lines)============
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from matplotlib.lines import Line2D

DPI = 500
C2  = pd.read_csv(rf'{DATA_ROOT}/new_climate/_proj_main/tables'
                  r'\C2_hazard_annual_series_ref2021_2040.csv')

CUTS = {'wave': [1.25, 2.5, 4.0, 6.0], 'wind': [8.0, 10.8, 13.9, 17.2]}
RAMP = {'wave': plt.cm.Blues(np.linspace(.42, .95, 4)),
        'wind': plt.cm.Oranges(np.linspace(.42, .95, 4))}
UNIT = {'wave': 'm', 'wind': 'm s$^{-1}$'}
SYM  = {'wave': '$H_s$', 'wind': '$U_{10}$'}
DASH = {'ACCESS': '-', 'ECEARTH': (0, (4, 1.6))}
MLAB = {'ACCESS': 'ACCESS-CM2', 'ECEARTH': 'EC-Earth3'}
ROLL = 11        # smoothing window; 0 = plot raw annual values

def series_panel(kind, ssp, annot=True, figsize=(5.2, 4.2)):
    fig, ax = plt.subplots(figsize=figsize, dpi=DPI)
    ax.axvspan(2021, 2040, color='#dfe7f2', alpha=.55, lw=0, zorder=0)
    ax.axvspan(2081, 2100, color='#e2e2e2', alpha=.60, lw=0, zorder=0)
    ax.axhline(0, color='#555555', lw=.7, zorder=2)
    for c, col in zip(CUTS[kind], RAMP[kind]):
        for mod in ('ACCESS', 'ECEARTH'):
            s = (C2[(C2.variable == kind) & (C2.scenario == ssp) &
                    (C2.model == mod) & (C2.cut == c)]
                 .set_index('year')['pct'].sort_index().loc[2021:2100])
            if ROLL:
                s = s.rolling(ROLL, center=True, min_periods=ROLL // 2 + 1).mean()
            ax.plot(s.index, s.values, ls=DASH[mod], color=col, lw=1.6, zorder=4)
    ax.set_xlim(2021, 2100); ax.set_ylim(-22, 12)
    ax.set_xticks([2021, 2040, 2060, 2080, 2100])
    for sp in ('top', 'right'): ax.spines[sp].set_visible(False)
    ax.grid(axis='y', alpha=.18, lw=.5)
    if annot:
        ax.text(2022, 10.3, 'Reference', fontsize=8, color='#44506a')
        ax.text(2099, 10.3, '2081-2100', fontsize=8, color='#4a4a4a', ha='right')
        ax.set_ylabel('Change in annual threshold\nexceedance days (%)', fontsize=10)
        base = {c: C2[(C2.variable == kind) & (C2.cut == c)].base.mean() for c in CUTS[kind]}
        ax.legend(handles=[Line2D([], [], color=col, lw=1.8,
                          label=f'{SYM[kind]} \u2265 {c:g} {UNIT[kind]} '
                                f'({base[c]:.0f} d yr$^{{-1}}$)')
                           for c, col in zip(CUTS[kind], RAMP[kind])]
                          + [Line2D([], [], color='#555', ls=DASH[m], lw=1.4, label=MLAB[m])
                             for m in DASH],
                  loc='lower left', fontsize=7, frameon=False, labelspacing=.35)
    plt.tight_layout(); plt.show()

for _k, _s in [('wave','ssp126'), ('wave','ssp585'), ('wind','ssp126'), ('wind','ssp585')]:
    series_panel(_k, _s, annot=True)
    series_panel(_k, _s, annot=False)


In [ ]:
# ==================== FIGURE C · 2 heat maps × with/without labels ====================
import os, numpy as np, pandas as pd, matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

DPI = 500
TB  = rf'{DATA_ROOT}/new_climate/_proj_main/tables'
B1  = pd.read_csv(os.path.join(TB, 'B1_risk_region_ref2021_2040.csv'))
B2  = pd.read_csv(os.path.join(TB, 'B2_risk_total_ref2021_2040.csv'))

SLICE, WEIGHT, LEVEL = '2081-2100', 'w_trf', 'SEA'
SEASONS = ['DJF', 'MAM', 'JJA', 'SON']
SSPS    = ['ssp126', 'ssp585']
ROWS    = [('NW_Europe','NWE'), ('NW_Atlantic','NWA'), ('Med_Black','MBS'),
           ('Asia','ESA'), ('Indian_Ocean_N','NIO'), ('__ALL__','All\nregions')]
COEF    = {'wave': 'RR_rq1_wave', 'wind': 'RR_rq1_wind'}
SYM     = {'wave': '$H_S$', 'wind': '$U_{10}$'}
CBLAB   = 'Projected change in accident occurrence rate (%)'
VMAX    = 8

def grab(kind):
    f = lambda d: d[(d.slice == SLICE) & (d.weight == WEIGHT) &
                    (d.level == LEVEL) & (d.coef == COEF[kind])]
    r, dot = np.full((len(ROWS), 8), np.nan), np.zeros((len(ROWS), 8), bool)
    for i, (reg, _) in enumerate(ROWS):
        d = f(B2) if reg == '__ALL__' else f(B1)[f(B1).region == reg]
        for j, (sea, ssp) in enumerate([(s, p) for s in SEASONS for p in SSPS]):
            v = d[(d.season == sea) & (d.scenario == ssp)].set_index('model')['pct']
            if len(v) == 2:
                r[i, j] = v.mean(); dot[i, j] = np.sign(v.iloc[0]) == np.sign(v.iloc[1])
    return r, dot

def risk_panel(kind, annot=True, cbar=False, figsize=(8.0, 5.0)):
    R, DOT = grab(kind)
    fig, ax = plt.subplots(figsize=figsize, dpi=DPI)
    im = ax.imshow(R, cmap='RdBu_r', vmin=-VMAX, vmax=VMAX, aspect='auto')
    for i in range(R.shape[0]):
        for j in range(R.shape[1]):
            if not np.isfinite(R[i, j]): continue
            ax.text(j, i, f'{R[i, j]:+.1f}', ha='center', va='center', fontsize=9.5,
                    color='white' if abs(R[i, j]) > .62 * VMAX else '#16202a')
            if DOT[i, j]:
                ax.plot(j + .36, i - .30, 'o', ms=3.4, color='#111111')
    for j, (sea, ssp) in enumerate([(s, p) for s in SEASONS for p in SSPS]):
        ax.add_patch(Rectangle((j - .47, -.47), .94, len(ROWS) - .06, fill=False,
                               ls=(0, (3, 2)), lw=1.1,
                               ec='#2f6fb0' if ssp == 'ssp126' else '#c0392b',
                               clip_on=False, zorder=5))
    ax.axhline(len(ROWS) - 1.5, color='#222222', lw=1.1)
    ax.set_xticks(np.arange(.5, 8, 2)); ax.set_yticks(range(len(ROWS)))
    ax.tick_params(length=0)
    if annot:
        ax.set_xticklabels(SEASONS, fontsize=11)
        ax.set_yticklabels([lab for _, lab in ROWS], fontsize=10.5, fontweight='bold')
        ax.set_title(SYM[kind], fontsize=13, pad=10)
    else:
        ax.set_xticklabels([]); ax.set_yticklabels([])
    for sp in ax.spines.values(): sp.set_visible(False)
    if cbar:
        cb = fig.colorbar(im, ax=ax, orientation='horizontal', fraction=.06,
                          pad=.10, aspect=34, extend='both')
        if annot: cb.set_label(CBLAB, fontsize=9)
        cb.ax.tick_params(labelsize=8); cb.outline.set_linewidth(.5)
    plt.tight_layout(); plt.show()

def risk_colorbar_only(figsize=(7.5, .8)):
    fig, ax = plt.subplots(figsize=figsize, dpi=DPI)
    cb = fig.colorbar(plt.cm.ScalarMappable(norm=plt.Normalize(-VMAX, VMAX),
                                            cmap='RdBu_r'),
                      cax=ax, orientation='horizontal', extend='both')
    cb.set_label(CBLAB, fontsize=10); cb.ax.tick_params(labelsize=9)
    cb.outline.set_linewidth(.5)
    fig.legend(handles=[plt.Line2D([], [], ls=(0,(3,2)), color='#2f6fb0', label='SSP1-2.6'),
                        plt.Line2D([], [], ls=(0,(3,2)), color='#c0392b', label='SSP5-8.5'),
                        plt.Line2D([], [], ls='', marker='o', ms=4, color='#111111',
                                   label='ACCESS-CM2 and EC-Earth3 agree on the direction')],
               loc='lower center', ncol=3, frameon=False, fontsize=9,
               bbox_to_anchor=(.5, -1.9))
    plt.show()

for _k in ['wave', 'wind']:
    risk_panel(_k, annot=True)
    risk_panel(_k, annot=False)
risk_colorbar_only()
